# vLLM Semantic Router on agentgateway — kind demo

Clients always send `"model": "auto"`. An ExtProc service (vLLM Semantic Router) classifies the prompt and **rewrites the request body** to pick the right LoRA adapter. Agentgateway forwards the mutated body. The router decides, the gateway enforces.

```mermaid
flowchart LR
  C[Client<br/>model: auto] --> GW(agentgateway<br/>:80)
  GW -.1 classify prompt.-> SR[vLLM Semantic Router<br/>:50051]
  SR -.2 rewrite body.-> GW
  GW -->|3 forward<br/>model: math-expert| VLLM
  subgraph VLLM["vLLM simulator (base-model)"]
    direction TB
    M[math-expert]
    L[law-expert]
    SC[science-expert]
    SO[social-expert]
    H[humanities-expert]
    G[general-expert]
  end

  classDef gw fill:#7C3AED,color:#fff,stroke:#5B21B6,stroke-width:3px
  classDef client fill:#DBEAFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
  classDef upstream fill:#D1FAE5,stroke:#10B981,color:#064E3B,stroke-width:2px
  classDef policy fill:#FEF3C7,stroke:#F59E0B,color:#78350F,stroke-width:2px
  classDef pick fill:#A7F3D0,stroke:#059669,color:#064E3B,stroke-width:3px
  class GW gw
  class C client
  class L,SC,SO,H,G upstream
  class M pick
  class SR policy
```

**What we'll cover:**
1. Show the data path that's already deployed (Gateway, AgentgatewayBackend, HTTPRoute, AgentgatewayPolicy)
2. Send one request and read the router's decision from `x-vsr-*` response headers
3. Sweep five categories — distinct adapters per category proves the body rewrite landed
4. Bypass test — delete the ExtProc policy, watch the same request fail, re-apply
5. Tail the router's decision logs

## Setup

Uses the existing `kind-vllm-sr` cluster from [`solo-demos/vllm-semantic-router-agentgateway-kind`](https://github.com/tjorourke/solo-demos/tree/main/vllm-semantic-router-agentgateway-kind). To bring it up from scratch: `./scripts/quick.sh up` in that folder.

We pin `KUBECTL_CONTEXT=kind-vllm-sr` so the rest of this notebook can stay on whichever cluster the user happens to be on globally.

In [ ]:
export KUBECTL_CONTEXT=kind-vllm-sr
kubectl --context $KUBECTL_CONTEXT cluster-info | head -1

echo
echo '── data-path objects (gateway / route / backend / extproc policy) ──'
kubectl --context $KUBECTL_CONTEXT -n agentgateway-system \
  get gateway,httproute,agentgatewaybackend,agentgatewaypolicy

echo
echo '── workloads ──'
kubectl --context $KUBECTL_CONTEXT -n agentgateway-system get pods
kubectl --context $KUBECTL_CONTEXT -n default              get deploy,svc

## 1. The data path — four resources

Four shapes wire the whole thing together. They're already applied; the cells below print them straight from the cluster so you can read them in place.

| Resource | What it does |
|---|---|
| `Gateway` (`vllm-gateway`) | OSS `agentgateway` GatewayClass listener on `:80` |
| `AgentgatewayBackend` (`semantic-router-vllm`) | AI backend with `provider.openai: {}` and **no model name** — the body the router writes wins |
| `HTTPRoute` | Send everything on the gateway to the AI backend |
| `AgentgatewayPolicy` (`semantic-router-extproc`) | Attach the router as ExtProc with `processingOptions.allowModeOverride: true` — this is the line that makes body rewriting actually take effect |

In [ ]:
for kind in gateway/vllm-gateway \
            agentgatewaybackend/semantic-router-vllm \
            httproute/semantic-router-vllm \
            agentgatewaypolicy/semantic-router-extproc; do
  echo "───── $kind ─────"
  kubectl --context $KUBECTL_CONTEXT -n agentgateway-system get $kind -o yaml \
    | yq 'del(
        .metadata.managedFields,
        .metadata.creationTimestamp, .metadata.uid,
        .metadata.resourceVersion, .metadata.generation,
        .metadata.annotations."kubectl.kubernetes.io/last-applied-configuration",
        .status
      )'
done

## 2. One request — read the router's decision

The router surfaces its decision as response headers:

- `x-vsr-selected-category` — the domain it classified into (`math`, `law`, `biology`, ...)
- `x-vsr-selected-model` — the LoRA adapter it picked (`math-expert`, `law-expert`, ...)

This is authoritative. The simulator also echoes the model in the JSON body — but the header is the cleanest proof that the router ran.

In [ ]:
# Kill ANY stale port-forward on :18080 (e.g. another demo's keycloak) and start fresh.
pkill -f 'port-forward.*18080:' 2>/dev/null; sleep 1
kubectl --context $KUBECTL_CONTEXT -n agentgateway-system \
  port-forward svc/vllm-gateway 18080:80 >/tmp/vllm-pf.log 2>&1 &
for _ in $(seq 1 10); do grep -q 'Forwarding from' /tmp/vllm-pf.log 2>/dev/null && break; sleep 1; done
head -1 /tmp/vllm-pf.log
echo "gateway → http://localhost:18080"

echo
echo '===== POST /v1/chat/completions   model: auto ====='
curl -sS -D /tmp/h -o /tmp/b -w 'HTTP %{http_code}\n' \
  -X POST http://localhost:18080/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"auto","messages":[{"role":"user","content":"What is the derivative of x^3?"}],"max_tokens":24,"temperature":0}'

echo
echo '===== router decision (x-vsr-* response headers) ====='
grep -i '^x-vsr' /tmp/h | sort

echo
echo '===== model the simulator actually served ====='
jq '{model, content: .choices[0].message.content}' /tmp/b 2>/dev/null || cat /tmp/b

## 3. Category sweep — distinct adapters prove the body rewrite landed

Same request shape (`"model": "auto"`), different prompts in different categories. If the router got a different adapter per category — and the gateway forwarded the mutated body — we see distinct `x-vsr-selected-model` values.

Maps from [`scripts/test.sh`](https://github.com/tjorourke/solo-demos/blob/main/vllm-semantic-router-agentgateway-kind/scripts/test.sh):

| Prompt | Expected category | Expected adapter |
|---|---|---|
| derivative of x³ | `math` | `math-expert` |
| elements of a valid contract | `law` | `law-expert` |
| mRNA vaccines | `health` | `science-expert` |
| pricing a SaaS product | `business` | `social-expert` |
| fall of the Roman Republic | `history` | `humanities-expert` |

In [ ]:
ask() {
  local label="$1" prompt="$2" hdr cat model
  hdr=$(curl -sS -D - -o /dev/null -X POST http://localhost:18080/v1/chat/completions \
    -H 'Content-Type: application/json' \
    -d "{\"model\":\"auto\",\"messages\":[{\"role\":\"user\",\"content\":\"${prompt}\"}],\"max_tokens\":16,\"temperature\":0}")
  cat=$(  printf '%s' "$hdr" | awk -F': ' 'tolower($1)=="x-vsr-selected-category"{print $2}' | tr -d '\r')
  model=$(printf '%s' "$hdr" | awk -F': ' 'tolower($1)=="x-vsr-selected-model"{print $2}'   | tr -d '\r')
  printf '  %-9s → category=%-12s adapter=%s\n' "$label" "${cat:-<none>}" "${model:-<none>}"
}

ask math     'What is the derivative of x^3?'
ask law      'What are the elements of a valid contract?'
ask biology  'How do mRNA vaccines trigger an immune response?'
ask business 'How should a startup price a SaaS product?'
ask history  'What caused the fall of the Roman Republic?'

## 4. Bypass test — pull the policy, watch it fail, put it back

When the ExtProc policy is gone the router never sees the request. `"model": "auto"` reaches the simulator untouched — and the simulator doesn't know a model named `auto`. The request fails. Re-applying the policy restores routing.

This cell is the most useful one in the demo: it's what proves the router is in the data path, not just sitting next to it.

In [ ]:
echo '===== with policy → 200 + adapter chosen ====='
curl -sS -o /dev/null -w 'HTTP %{http_code}  ' http://localhost:18080/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"auto","messages":[{"role":"user","content":"derivative of x^3"}],"max_tokens":8,"temperature":0}'
curl -sS -D - -o /dev/null -X POST http://localhost:18080/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"auto","messages":[{"role":"user","content":"derivative of x^3"}],"max_tokens":8,"temperature":0}' \
  | grep -i '^x-vsr' | sort

echo
echo '===== detach ExtProc policy and retry ====='
kubectl --context $KUBECTL_CONTEXT -n agentgateway-system \
  get agentgatewaypolicy semantic-router-extproc -o yaml > /tmp/extproc-policy.yaml
kubectl --context $KUBECTL_CONTEXT -n agentgateway-system \
  delete agentgatewaypolicy semantic-router-extproc
sleep 3
curl -sS -o /tmp/body -w 'HTTP %{http_code}\n' http://localhost:18080/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"auto","messages":[{"role":"user","content":"derivative of x^3"}],"max_tokens":8,"temperature":0}'
echo '── body ──'
head -c 300 /tmp/body; echo

echo
echo '===== re-apply policy ====='
kubectl --context $KUBECTL_CONTEXT apply -f /tmp/extproc-policy.yaml
sleep 3
curl -sS -o /dev/null -w 'HTTP %{http_code}\n' http://localhost:18080/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"auto","messages":[{"role":"user","content":"derivative of x^3"}],"max_tokens":8,"temperature":0}'

## 5. Router decision logs

The semantic-router pod logs every classification — the category it picked, the adapter it chose, and which plugins fired (system-prompt, semantic-cache, etc.).

In [ ]:
# Pretty-print the routing_decision events (one per request).
kubectl --context $KUBECTL_CONTEXT -n agentgateway-system \
  logs deploy/semantic-router --tail=400 2>/dev/null \
  | grep '"msg":"routing_decision"' \
  | tail -8 \
  | jq -c '{ts, original_model, selected_model, decision, reasoning_enabled, latency_ms: .routing_latency_ms}' 

## Wrap-up

Clients send `"model": "auto"` to one endpoint. A gRPC ExtProc service classifies the prompt and rewrites the request body to pick a LoRA adapter on a single vLLM backend. Same Kubernetes Gateway API model as any agentgateway demo, with two specifics:

- The AI backend's `provider.openai: {}` has **no model name** — the body the router writes wins.
- The ExtProc policy carries `processingOptions.allowModeOverride: true` — only the OSS upstream agentgateway CRD exposes that knob, and it's the line that makes body rewriting take effect.

Real LoRA adapters slot in by swapping the simulator for `vllm serve` with `--enable-lora --lora-modules name=/path`. The contract that makes routing work is name matching — adapter names must be identical across vLLM, the router's category→adapter map, and `GET /v1/models`.

## Cleanup (optional)

Kills the gateway port-forward. The kind cluster, the gateway, the router, and the vLLM simulator all stay up — this notebook is designed to be re-run on the same cluster many times. To tear the whole lab down, run `./scripts/quick.sh teardown` from the [`vllm-semantic-router-agentgateway-kind`](https://github.com/tjorourke/solo-demos/tree/main/vllm-semantic-router-agentgateway-kind) folder.

In [ ]:
pkill -f 'port-forward.*svc/vllm-gateway.*18080:80' 2>/dev/null
echo 'port-forward stopped'